<a href="https://colab.research.google.com/github/AI453/EWARS/blob/main/EWARS_pilot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from google.colab import drive
import numpy as np
drive.mount('/content/drive')

!pip install xlsx2csv

# House Keeping - All input files have to be saved on the drive of the person running the file. Ideally files will get updated daily
# ipt refers to the input file - contains the static thresholds
ipt_p = pd.read_excel('/content/drive/MyDrive/EWARS/Thresholds_static_p.xlsx')
ipt_l = pd.read_excel('/content/drive/MyDrive/EWARS/Thresholds_static_l.xlsx')
daily_l_df = pd.read_csv('/content/drive/MyDrive/EWARS/t_mp_lform_hfwise_ll_jan22tojun22.csv')
daily_p_df = pd.read_csv('/content/drive/MyDrive/EWARS/t_mp_pform_hfwise_ll_jan22tojun22.csv')

### Static Thresholds

In [ ]:
# P Form
# Function to loop through cols while appyling corresponding threshold
def apply_threshold(df, disease, threshold):
    # Apply the threshold condition to the dataframe for the given disease.

    if disease == 'Dysentry':
        threshold_cond = df[disease] >= threshold
        combined_cond = threshold_cond | df['Acute Diarrhoeal Disease_deaths'] >= threshold | df['Dysentry_deaths'] >= threshold
    else:
        combined_cond = df[disease] >= threshold

    # Apply the combined condition to set the values in the new column
    df[f'{disease}_alert_district'] = df.groupby(['dt', 'districtname'])[disease].transform(lambda x: (x >= threshold).astype(int))
    df[f'{disease}_alert_subdistrict'] = df.groupby(['dt', 'subdistrictname'])[disease].transform(lambda x: (x >= threshold).astype(int))
    df[f'{disease}_alert_facility'] = combined_cond.astype(int)

# Iterate through the "Disease" and "Threshold" columns in df_excel
for index, row in ipt_p.iterrows():
    disease = row['Disease']
    threshold = row['Threshold']

    # Check if the disease column exists in df
    if disease in daily_p_df.columns:
        apply_threshold(daily_p_df, disease, threshold)

In [ ]:
# L Form - or change the df argument in the if loop above, and the ipt file in the for loop
# Function to loop through cols while appyling corresponding threshold
def apply_threshold(df, disease, threshold):
    # Apply the threshold condition to the dataframe for the given disease.

    if disease == 'Dysentry':
        threshold_cond = df[disease] >= threshold
        combined_cond = threshold_cond | df['Acute Diarrhoeal Disease_deaths'] >= threshold | df['Dysentry_deaths'] >= threshold
    else:
        combined_cond = df[disease] >= threshold

    # Apply the combined condition to set the values in the new column
    df[f'{disease}_alert_district'] = df.groupby(['dt', 'districtname'])[disease].transform(lambda x: (x >= threshold).astype(int))
    df[f'{disease}_alert_subdistrict'] = df.groupby(['dt', 'subdistrictname'])[disease].transform(lambda x: (x >= threshold).astype(int))
    df[f'{disease}_alert_facility'] = combined_cond.astype(int)

# Iterate through the "Disease" and "Threshold" columns in df_excel
for index, row in ipt_l.iterrows():
    disease = row['Disease']
    threshold = row['Threshold']

    # Check if the disease column exists in df
    if disease in daily_l_df.columns:
        apply_threshold(daily_l_df, disease, threshold)

In [ ]:
alert_columns = [col for col in daily_p_df.columns if col.endswith("_alert_district")]
alert_columns_subdistrict = [col for col in daily_p_df.columns if col.endswith("_alert_subdistrict")]
alert_columns_facility = [col for col in daily_p_df.columns if col.endswith("_alert_facility")]
static_columns = daily_p_df.columns[:18]

# Create new DataFrames for the alert columns and static columns
df_alert_district= daily_p_df[alert_columns]
df_alert_subdistrict= daily_p_df[alert_columns_subdistrict]
df_alert_facility= daily_p_df[alert_columns_facility]
df_static = daily_p_df[static_columns]

# Concatenate the two DataFrames along the columns axis (axis=1)
df_result_district = pd.concat([df_static, df_alert_district], axis=1)
df_result_subdistrict = pd.concat([df_static, df_alert_subdistrict], axis=1)
df_result_facility = pd.concat([df_static, df_alert_facility], axis=1)

# Reduce Redunduncies - check spelling
col_names = df_result_district.columns[18:]
p_result_district = df_result_district.groupby(['dt', 'districtname'])[col_names].agg('min').reset_index()
col_names = df_result_subdistrict.columns[18:]
p_result_subdistrict = df_result_subdistrict.groupby(['dt', 'districtname', 'subdistrictname'])[col_names].agg('min').reset_index()
col_names = df_result_facility.columns[18:]
p_result_hf = df_result_facility.groupby(['dt', 'districtname', 'subdistrictname', 'hfname'])[col_names].agg('min').reset_index()

In [ ]:
#export to csv
path = ('static_p_district.csv')
p_result_district.to_csv(path, index = False)
path = ('static_p_subdistrict.csv')
p_result_subdistrict.to_csv(path, index = False)
path = ('static_p_facility.csv')
p_result_hf.to_csv(path, index = False)

In [ ]:
alert_columns = [col for col in daily_l_df.columns if col.endswith("_alert_district")]
alert_columns_subdistrict = [col for col in daily_l_df.columns if col.endswith("_alert_subdistrict")]
alert_columns_facility = [col for col in daily_l_df.columns if col.endswith("_alert_facility")]
static_columns = daily_l_df.columns[:18]

# Create new DataFrames for the alert columns and static columns
df_alert_district= daily_l_df[alert_columns]
df_alert_subdistrict= daily_l_df[alert_columns_subdistrict]
df_alert_facility= daily_l_df[alert_columns_facility]
df_static = daily_l_df[static_columns]

# Concatenate the two DataFrames along the columns axis (axis=1)
df_result_district = pd.concat([df_static, df_alert_district], axis=1)
df_result_subdistrict = pd.concat([df_static, df_alert_subdistrict], axis=1)
df_result_facility = pd.concat([df_static, df_alert_facility], axis=1)

# Reduce Redunduncies - check spelling
col_names = df_result_district.columns[18:]
l_result_district = df_result_district.groupby(['dt', 'districtname'])[col_names].agg('min').reset_index()
col_names = df_result_subdistrict.columns[18:]
l_result_subdistrict = df_result_subdistrict.groupby(['dt', 'districtname', 'subdistrictname'])[col_names].agg('min').reset_index()
col_names = df_result_facility.columns[18:]
l_result_hf = df_result_facility.groupby(['dt', 'districtname', 'subdistrictname', 'hfname'])[col_names].agg('min').reset_index()

In [ ]:
#export to csv
path = ('static_l_district.csv')
l_result_district.to_csv(path, index = False)
path = ('static_l_subdistrict.csv')
l_result_subdistrict.to_csv(path, index = False)
path = ('static_l_facility.csv')
l_result_hf.to_csv(path, index = False)

### Dynamic Tresholds

In [ ]:
daily_l_df = pd.read_csv('/content/drive/MyDrive/EWARS/t_mp_lform_hfwise_ll_jan22tojun22.csv')
daily_p_df = pd.read_csv('/content/drive/MyDrive/EWARS/t_mp_pform_hfwise_ll_jan22tojun22.csv')

In [ ]:
columns_to_keep = daily_l_df.columns[:18].to_list()
dengue_df = daily_l_df[columns_to_keep + ['Dengue_tested', 'Dengue_positive']]
malaria_df = daily_l_df[columns_to_keep + ['Malaria_tested', 'Malaria_positive']]
chikungunya_df = daily_l_df[columns_to_keep + ['Chikungunya_tested', 'Chikungunya_positive']]
typhoid_df = daily_l_df[columns_to_keep + ['Typhoid_tested', 'Typhoid_positive']]
covid_df = daily_l_df[columns_to_keep + ['COVID-19_tested', 'COVID-19_positive']]
influenza_df = daily_l_df[columns_to_keep + ['Influenza_tested', 'Influenza_positive']]
# fever_with_rash_df = daily_l_df[columns_to_keep + ['Fever with Rash_c', 'Fever with Rash_deaths']]

In [ ]:
def calculate_alert_per_disease(pivot, disease_prefix):
    # Sort the DataFrame by date
    pivot.sort_values(by='dt', inplace=True)

    # Identify columns related to the current disease
    positive_column = f"{disease_prefix}_positive"
    tested_column = f"{disease_prefix}_tested"

    # Calculate the rolling sum of 'Value' for the last 3 months for tested cases
    pivot['RollingSumPositive'] = pivot.groupby(['subdistrictname', 'dt'])[positive_column].rolling(window=30, min_periods=1).sum().reset_index(level=['subdistrictname', 'dt'])[positive_column]
    pivot['RollingSumPositivem1'] = pivot.groupby(['subdistrictname', 'dt'])[positive_column].shift(1).rolling(window=30, min_periods=1).sum()
    pivot['RollingSumPositivem2'] = pivot.groupby(['subdistrictname', 'dt'])[positive_column].shift(31).rolling(window=30, min_periods=1).sum()
    pivot['RollingSumPositivem3'] = pivot.groupby(['subdistrictname', 'dt'])[positive_column].shift(61).rolling(window=30, min_periods=1).sum()
    pivot['RollingSum3months'] = pivot.groupby(['subdistrictname', 'dt'])[positive_column].shift(1).rolling(window=90, min_periods=1).sum()

    pivot['RollingSumTested'] = pivot.groupby(['subdistrictname', 'dt'])[tested_column].rolling(window=30, min_periods=1).sum().reset_index(level=['subdistrictname', 'dt'])[tested_column]
    pivot['RollingSumTestedm1'] = pivot.groupby(['subdistrictname', 'dt'])[tested_column].shift(1).rolling(window=30, min_periods=1).sum()
    pivot['RollingSumTestedm2'] = pivot.groupby(['subdistrictname', 'dt'])[tested_column].shift(31).rolling(window=30, min_periods=1).sum()
    pivot['RollingSumTestedm3'] = pivot.groupby(['subdistrictname', 'dt'])[tested_column].shift(61).rolling(window=30, min_periods=1).sum()

    # Initialize the 'alert_subdistrict' column to None
    pivot['alert_subdistrict1'] = None
    pivot['alert_threshold1'] = 0
    pivot['alert_subdistrict2'] = None
    pivot['alert_threshold2'] = 0
    pivot['alert_subdistrict3'] = None
    pivot['alert_threshold3'] = 0

    k = 2.0

    # Condition 1: k-fold rise in cases
    condition_1 = (pivot['RollingSumPositive'] > k * pivot[['RollingSumPositivem1', 'RollingSumPositivem2', 'RollingSumPositivem3']].apply(np.max, axis=1))
    pivot.loc[condition_1, 'alert_subdistrict1'] = 'Two-fold Rise in Cases Cases'
    pivot.loc[condition_1, 'alert_threshold1'] = 1

    # Condition 2: Even a single case where it was not present for 3 months
    condition_2 = (pivot[positive_column] > 0) & (pivot['RollingSum3months'] == 0)
    pivot.loc[condition_2, 'alert_subdistrict2'] = 'Single Case in an Area Where Disease Was not present'
    pivot.loc[condition_2, 'alert_threshold2'] = 2
    # Condition 3: Slide Positivity
    k = 2.0
    condition_3 = (
        (
            (pivot['RollingSumTested'] != 0) & (pivot['RollingSumTestedm1'] != 0) & # Exclude cases where the denominator is zero
            ((pivot['RollingSumPositive'] / pivot['RollingSumTested']) > k * (pivot['RollingSumPositivem1'] / pivot['RollingSumTestedm1']))
        ) |
        (
            (pivot['RollingSumTested'] != 0) & (pivot['RollingSumTestedm2'] != 0) &
            ((pivot['RollingSumPositive'] / pivot['RollingSumTested']) > k * (pivot['RollingSumPositivem2'] / pivot['RollingSumTestedm2']))
        ) |
        (
            (pivot['RollingSumTested'] != 0) & (pivot['RollingSumTestedm3'] != 0) &
            ((pivot['RollingSumPositive'] / pivot['RollingSumTested']) > k * (pivot['RollingSumPositivem3'] / pivot['RollingSumTestedm3']))
        )
    )

    pivot.loc[condition_3, 'alert_subdistrict3'] = 'Slide Positivity'
    pivot.loc[condition_3, 'alert_threshold3'] = 3
    pivot.drop(['RollingSumTested', 'RollingSumTestedm1', 'RollingSumTestedm2', 'RollingSumTestedm3',
                'RollingSumPositive', 'RollingSumPositivem1', 'RollingSumPositivem2', 'RollingSumPositivem3'], axis=1, inplace=True)

    return pivot

# Example usage for Dengue
l_result_dengue_df = calculate_alert_per_disease(dengue_df.copy(), "Dengue")
l_result_malaria_df = calculate_alert_per_disease(malaria_df.copy(), "Malaria")
l_result_chikungunya_df = calculate_alert_per_disease(chikungunya_df.copy(), "Chikungunya")
l_result_covid_df = calculate_alert_per_disease(covid_df.copy(), "COVID-19")
l_result_typhoid_df = calculate_alert_per_disease(typhoid_df.copy(), "Typhoid")
l_result_influenza_df = calculate_alert_per_disease(influenza_df.copy(), "Influenza")

In [ ]:
#Export L form dynamic results
col_indices = l_result_dengue_df.columns[18:]
l_result_dengue_df = l_result_dengue_df.groupby(['dt', 'districtname', 'subdistrictname'])[col_indices].agg('min').reset_index()
col_indices = l_result_malaria_df.columns[18:]
l_result_malaria_df = l_result_malaria_df.groupby(['dt', 'districtname', 'subdistrictname'])[col_indices].agg('min').reset_index()
col_indices = l_result_chikungunya_df.columns[18:]
l_result_chikungunya_df = l_result_chikungunya_df.groupby(['dt', 'districtname', 'subdistrictname'])[col_indices].agg('min').reset_index()
col_indices = l_result_typhoid_df.columns[18:]
l_result_typhoid_df = l_result_typhoid_df.groupby(['dt', 'districtname', 'subdistrictname'])[col_indices].agg('min').reset_index()
col_indices = l_result_covid_df.columns[18:]
l_result_covid_df = l_result_covid_df.groupby(['dt', 'districtname', 'subdistrictname'])[col_indices].agg('min').reset_index()
col_indices = l_result_influenza_df.columns[18:]
l_result_influenza_df = l_result_influenza_df.groupby(['dt', 'districtname', 'subdistrictname'])[col_indices].agg('min').reset_index()

In [ ]:
#export to csv
path = ('l_dengue.csv')
l_result_dengue_df.to_csv(path, index = False)
path = ('l_malaria.csv')
l_result_malaria_df.to_csv(path, index = False)
path = ('l_chikungunya.csv')
l_result_chikungunya_df.to_csv(path, index = False)
path = ('l_typhoid.csv')
l_result_typhoid_df.to_csv(path, index = False)
path = ('l_covid.csv')
l_result_covid_df.to_csv(path, index = False)
path = ('l_influenza.csv')
l_result_influenza_df.to_csv(path, index = False)
# path = ('fever_with_rash.csv')
# result_fever_with_rash_df.to_csv(path, index = False)

In [ ]:
p_dengue_df = daily_p_df[columns_to_keep + ['Dengue_cases', 'Dengue_deaths']]
p_malaria_df = daily_p_df[columns_to_keep + ['Malaria_cases', 'Malaria_deaths']]
p_chikungunya_df = daily_p_df[columns_to_keep + ['Chikungunya_cases', 'Chikungunya_deaths']]
p_typhoid_df = daily_p_df[columns_to_keep + ['Typhoid_cases', 'Typhoid_deaths']]
p_fever_with_rash_df = daily_p_df[columns_to_keep + ['Fever with Rash_cases', 'Fever with Rash_deaths']]

In [ ]:
def calculate_alert_per_disease_p(pivot, disease_prefix):
    # Sort the DataFrame by date
    pivot.sort_values(by='dt', inplace=True)

    # Identify columns related to the current disease
    positive_column = f"{disease_prefix}_cases"
    tested_column = f"{disease_prefix}_deaths"

    # Calculate the rolling sum of 'Value' for the last 3 months for tested cases
    pivot['RollingSumCases'] = pivot.groupby(['subdistrictname', 'dt'])[positive_column].rolling(window=30, min_periods=1).sum().reset_index(level=['subdistrictname', 'dt'])[positive_column]
    pivot['RollingSumCasesm1'] = pivot.groupby(['subdistrictname', 'dt'])[positive_column].shift(1).rolling(window=30, min_periods=1).sum()
    pivot['RollingSumCasesm2'] = pivot.groupby(['subdistrictname', 'dt'])[positive_column].shift(31).rolling(window=30, min_periods=1).sum()
    pivot['RollingSumCasesm3'] = pivot.groupby(['subdistrictname', 'dt'])[positive_column].shift(61).rolling(window=30, min_periods=1).sum()
    pivot['RollingSum3months'] = pivot.groupby(['subdistrictname', 'dt'])[positive_column].shift(1).rolling(window=90, min_periods=1).sum()

    # Initialize the 'alert_subdistrict' column to None
    pivot['alert_threshold'] = 0
    pivot['alert_threshold1'] = 0

  #  # Condition 1: Two-fold rise in cases, min of 3
  #   pivot.loc[(pivot['RollingSumCases'] > 2.0*pivot['RollingSumCasesm1']) |
  #          (pivot['RollingSumCases'] > 2.0*pivot['RollingSumCasesm2']) |
  #          (pivot['RollingSumCases'] > 2.0*pivot['RollingSumCasesm3']),
  #          'alert_subdistrict'] = 'Two-fold Rise in Cases Cases'
  #   pivot.loc[(pivot['RollingSumCases'] > 2.0*pivot['RollingSumCasesm1']) |
  #          (pivot['RollingSumCases'] > 2.0*pivot['RollingSumCasesm2']) |
  #          (pivot['RollingSumCases'] > 2.0*pivot['RollingSumCasesm3']),
  #          'alert_threshold'] = 1

    # Max of m1/m2/m3 test run
    # Calculate the maximum of RollingSumCasesm1, RollingSumCasesm2, and RollingSumCasesm3 for each row
    pivot['max_m_values'] = pivot[['RollingSumCasesm1', 'RollingSumCasesm2', 'RollingSumCasesm3']].apply(lambda x: np.max(x), axis=1)

    # Set alert_threshold and alert_subdistrict based on the condition
    pivot.loc[(pivot['RollingSumCases'] > 2 * pivot['max_m_values']), 'alert_threshold'] = 1
    pivot.loc[(pivot['RollingSumCases'] > 2 * pivot['max_m_values']), 'alert_subdistrict'] = "Two-Fold Rise"
    # Drop the temporary column max_m_values if needed
    pivot.drop('max_m_values', axis=1, inplace=True)
    # pivot['alert_subdistrict1'] = np.where((pivot[tested_column] > 0), 'Death by Disease', '')
    # pivot['alert_threshold1'] = np.where((pivot[tested_column] > 0), 1, 0)
    pivot.loc[(pivot[tested_column] > 0),
           'alert_threshold1'] = 1
    pivot.loc[(pivot[tested_column] > 0),
           'alert_subdistrict1'] = "Death by Disease"

    # Condition 4: C-SUM
    ## to do
    # Drop temporary columns
    pivot.drop(['RollingSumCases', 'RollingSumCasesm1','RollingSumCasesm2',
             'RollingSumCasesm3'], axis=1, inplace=True)
    return pivot

p_result_dengue_df = calculate_alert_per_disease_p(p_dengue_df.copy(), "Dengue")
p_result_malaria_df = calculate_alert_per_disease_p(p_malaria_df.copy(), "Malaria")
p_result_chikungunya_df = calculate_alert_per_disease_p(p_chikungunya_df.copy(), "Chikungunya")
p_result_typhoid_df = calculate_alert_per_disease_p(p_typhoid_df.copy(), "Typhoid")
p_result_feverwithrash_df = calculate_alert_per_disease_p(p_fever_with_rash_df.copy(), "Fever with Rash")

In [ ]:
#Export P form dynamic results
col_indices = p_result_dengue_df.columns[18:]
p_result_dengue_df = p_result_dengue_df.groupby(['dt', 'districtname', 'subdistrictname'])[col_indices].agg('min').reset_index()
col_indices = p_result_malaria_df.columns[18:]
p_result_malaria_df = p_result_malaria_df.groupby(['dt', 'districtname', 'subdistrictname'])[col_indices].agg('min').reset_index()
col_indices = p_result_chikungunya_df.columns[18:]
p_result_chikungunya_df = p_result_chikungunya_df.groupby(['dt', 'districtname', 'subdistrictname'])[col_indices].agg('min').reset_index()
col_indices = p_result_typhoid_df.columns[18:]
p_result_typhoid_df = p_result_typhoid_df.groupby(['dt', 'districtname', 'subdistrictname'])[col_indices].agg('min').reset_index()
col_indices = p_result_feverwithrash_df.columns[18:]
p_result_feverwithrash_df = p_result_feverwithrash_df.groupby(['dt', 'districtname', 'subdistrictname'])[col_indices].agg('min').reset_index()

In [ ]:
path = ('p_dengue.csv')
p_result_dengue_df.to_csv(path, index = False)
path = ('p_malaria.csv')
p_result_malaria_df.to_csv(path, index = False)
path = ('p_chikungunya.csv')
p_result_chikungunya_df.to_csv(path, index = False)
path = ('p_typhoid.csv')
p_result_typhoid_df.to_csv(path, index = False)
path = ('p_feverwithrash.csv')
p_result_feverwithrash_df.to_csv(path, index = False)